In [1]:
# ============================================================
# STEP 300 — LOAD ALL PRODUCTION ARTIFACTS
# ============================================================

from pathlib import Path
import pandas as pd

project_root = Path.cwd()

while not (project_root / "models").exists():
    if project_root.parent == project_root:
        raise RuntimeError(
            "Could not locate CODEZILLA project root."
        )
    project_root = project_root.parent

processed = (
    project_root
    / "data"
    / "processed"
)

# ------------------------------------------------------------
# Load artifacts
# ------------------------------------------------------------

ddos_features = pd.read_parquet(
    processed / "ddos_features.parquet"
)

ddos_inference = pd.read_parquet(
    processed / "ddos_inference.parquet"
)

c2_features = pd.read_parquet(
    processed / "c2_features.parquet"
)

dns_features = pd.read_parquet(
    processed / "dns_source_features.parquet"
)

encrypted_features = pd.read_parquet(
    processed / "encrypted_source_features.parquet"
)

print("✅ ALL ARTIFACTS LOADED")

print("\nDDoS:")
print(ddos_features.shape)

print("\nDDoS inference:")
print(ddos_inference.shape)

print("\nC2:")
print(c2_features.shape)

print("\nDNS:")
print(dns_features.shape)

print("\nEncrypted:")
print(encrypted_features.shape)

✅ ALL ARTIFACTS LOADED

DDoS:
(1949, 65)

DDoS inference:
(1949, 5)

C2:
(3020, 22)

DNS:
(40953, 25)

Encrypted:
(47507, 16)


In [2]:
# ============================================================
# STEP 301 — DATASET COMPATIBILITY CHECK
# ============================================================

def show_time_range(name, df):

    if "time_window" not in df.columns:
        print(f"{name}: no time_window column")
        return

    print(
        f"{name}: "
        f"{df['time_window'].min()} "
        f"-> "
        f"{df['time_window'].max()}"
    )


show_time_range(
    "DDoS",
    ddos_features
)

show_time_range(
    "C2",
    c2_features
)

show_time_range(
    "DNS",
    dns_features
)

show_time_range(
    "Encrypted",
    encrypted_features
)

DDoS: 2018-02-21 01:55:46 -> 2018-02-21 10:43:20
C2: 2011-08-11 09:49:35 -> 2011-08-11 14:01:10
DNS: 2011-08-10 09:46:50 -> 2011-08-10 15:54:00
Encrypted: 2011-08-10 09:46:50 -> 2011-08-10 15:54:00


In [3]:
# ============================================================
# STEP 302 — CODEZILLA FINAL ALERT SCHEMA
# ============================================================

from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional


@dataclass
class DetectorAlert:

    detector: str

    prediction: str

    score: float

    threat_class: str

    severity: str

    time_window: Any = None

    source: Optional[str] = None

    evidence: List[Dict[str, Any]] = None

    def to_dict(self):

        data = asdict(self)

        if data["evidence"] is None:
            data["evidence"] = []

        return data


@dataclass
class UnifiedAlert:

    prediction: str

    severity: str

    score: float

    primary_threat: Optional[str]

    source: Optional[str]

    time_window: Any

    detector_count: int

    threats: List[Dict[str, Any]]

    evidence: List[Dict[str, Any]]

    def to_dict(self):

        return asdict(self)


print("✅ Final alert schema created")

✅ Final alert schema created


In [4]:
# ============================================================
# STEP 303 — FINAL ALERT FUSION
# ============================================================

def fuse_detector_alerts(
    alerts,
    *,
    source=None,
    time_window=None,
):
    """
    Combine detector alerts belonging to the same event.

    This function does NOT merge unrelated historical
    datasets. The caller is responsible for event alignment.
    """

    active = [
        alert
        for alert in alerts
        if alert.get("prediction", "BENIGN") != "BENIGN"
    ]

    if not active:

        return UnifiedAlert(
            prediction="BENIGN",
            severity="LOW",
            score=0.0,
            primary_threat=None,
            source=source,
            time_window=time_window,
            detector_count=0,
            threats=[],
            evidence=[]
        ).to_dict()

    # Highest detector score
    active = sorted(
        active,
        key=lambda x: float(
            x.get("model_score", 0.0)
        ),
        reverse=True
    )

    # Combine independent evidence
    evidence = []

    for alert in active:

        evidence.extend(
            alert.get(
                "supporting_features",
                []
            )
        )

    # Severity ranking
    severity_rank = {
        "LOW": 1,
        "MEDIUM": 2,
        "HIGH": 3,
        "CRITICAL": 4,
    }

    severity = max(
        (
            str(
                a.get(
                    "severity",
                    "LOW"
                )
            ).upper()
            for a in active
        ),
        key=lambda x: severity_rank.get(
            x,
            0
        )
    )

    primary = active[0].get(
        "threat_class"
    )

    score = float(
        active[0].get(
            "model_score",
            0.0
        )
    )

    threats = []

    for alert in active:

        threats.append({
            "detector": alert.get(
                "detector"
            ),
            "threat_class": alert.get(
                "threat_class"
            ),
            "prediction": alert.get(
                "prediction"
            ),
            "score": float(
                alert.get(
                    "model_score",
                    0.0
                )
            ),
            "severity": alert.get(
                "severity",
                "LOW"
            ),
            "supporting_features": alert.get(
                "supporting_features",
                []
            ),
        })

    return UnifiedAlert(
        prediction="THREAT",
        severity=severity,
        score=round(
            score,
            4
        ),
        primary_threat=primary,
        source=source,
        time_window=time_window,
        detector_count=len(active),
        threats=threats,
        evidence=evidence,
    ).to_dict()


print("✅ Final fusion function created")

✅ Final fusion function created


In [5]:
# ============================================================
# STEP 304 — FINAL FUSION TEST
# ============================================================

demo_alerts = [

    {
        "detector": "ENCRYPTED_TRAFFIC",
        "prediction": "ENCRYPTED_THREAT",
        "model_score": 0.97,
        "threat_class": "ENCRYPTED_TRAFFIC",
        "severity": "HIGH",
        "supporting_features": [
            {
                "feature": "bytes_per_flow",
                "feature_value": 85562
            }
        ],
    },

    {
        "detector": "C2",
        "prediction": "C2",
        "model_score": 0.82,
        "threat_class": "C2",
        "severity": "HIGH",
        "supporting_features": [
            {
                "feature": "pair_repetition_ratio",
                "feature_value": 0.72
            }
        ],
    },
]

final_alert = fuse_detector_alerts(
    demo_alerts,
    source="147.32.84.165",
    time_window="2011-08-11 13:12:30",
)

print("FINAL CODEZILLA ALERT")
print("=====================")

import pprint

pprint.pp(
    final_alert
)

FINAL CODEZILLA ALERT
{'prediction': 'THREAT',
 'severity': 'HIGH',
 'score': 0.97,
 'primary_threat': 'ENCRYPTED_TRAFFIC',
 'source': '147.32.84.165',
 'time_window': '2011-08-11 13:12:30',
 'detector_count': 2,
 'threats': [{'detector': 'ENCRYPTED_TRAFFIC',
              'threat_class': 'ENCRYPTED_TRAFFIC',
              'prediction': 'ENCRYPTED_THREAT',
              'score': 0.97,
              'severity': 'HIGH',
              'supporting_features': [{'feature': 'bytes_per_flow',
                                       'feature_value': 85562}]},
             {'detector': 'C2',
              'threat_class': 'C2',
              'prediction': 'C2',
              'score': 0.82,
              'severity': 'HIGH',
              'supporting_features': [{'feature': 'pair_repetition_ratio',
                                       'feature_value': 0.72}]}],
 'evidence': [{'feature': 'bytes_per_flow', 'feature_value': 85562},
              {'feature': 'pair_repetition_ratio', 'feature_value': 0

In [6]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

while not (project_root / "data").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Project root not found")
    project_root = project_root.parent

processed = project_root / "data" / "processed"

artifacts = {
    "ddos_features": processed / "ddos_features.parquet",
    "ddos_inference": processed / "ddos_inference.parquet",
    "c2": processed / "c2_features.parquet",
    "dns": processed / "dns_source_features.parquet",
    "encrypted": processed / "encrypted_source_features.parquet",
}

for name, path in artifacts.items():
    print(
        f"{name}:",
        path.exists(),
        path
    )

ddos_features: True e:\CODEZILLA-SIH26145\data\processed\ddos_features.parquet
ddos_inference: True e:\CODEZILLA-SIH26145\data\processed\ddos_inference.parquet
c2: True e:\CODEZILLA-SIH26145\data\processed\c2_features.parquet
dns: True e:\CODEZILLA-SIH26145\data\processed\dns_source_features.parquet
encrypted: True e:\CODEZILLA-SIH26145\data\processed\encrypted_source_features.parquet


In [7]:
ddos = pd.read_parquet(
    artifacts["ddos_inference"]
)

c2 = pd.read_parquet(
    artifacts["c2"]
)

dns = pd.read_parquet(
    artifacts["dns"]
)

encrypted = pd.read_parquet(
    artifacts["encrypted"]
)

print(" Detector feeds loaded")

print("DDoS:", ddos.shape)
print("C2:", c2.shape)
print("DNS:", dns.shape)
print("Encrypted:", encrypted.shape)

 Detector feeds loaded
DDoS: (1949, 5)
C2: (3020, 22)
DNS: (40953, 25)
Encrypted: (47507, 16)


In [8]:
# ============================================================
# STEP 307 — EVENT SCOPES
# ============================================================

def build_ddos_alerts(ddos_df):
    """
    DDoS is GLOBAL_WINDOW scoped.
    It is not attributed to a source IP.
    """

    alerts = []

    for _, row in ddos_df.iterrows():

        if int(row["prediction"]) != 1:
            continue

        alerts.append({
            "scope": "GLOBAL_WINDOW",
            "detector": "DDoS",
            "time_window": row["time_window"],
            "source": None,
            "prediction": "THREAT",
            "score": float(row["model_score"]),
            "threat_class": "DDoS",
            "severity": row["severity"],
            "evidence": []
        })

    return alerts


ddos_alerts = build_ddos_alerts(ddos)

print(
    " DDoS alerts:",
    len(ddos_alerts)
)

 DDoS alerts: 1207


In [9]:
# ============================================================
# STEP 308 — SOURCE EVENT KEY
# ============================================================

def source_key(row):
    return (
        row.get("SrcAddr"),
        row.get("time_window")
    )


print(" Source event key ready")

 Source event key ready


In [10]:
# ============================================================
# STEP 309 — FINAL PRODUCT ALERT
# ============================================================

def make_final_alert(
    *,
    source,
    time_window,
    detector_results
):

    active = [
        r for r in detector_results
        if r["prediction"] != "BENIGN"
    ]

    if not active:

        return {
            "prediction": "BENIGN",
            "severity": "LOW",
            "score": 0.0,
            "source": source,
            "time_window": time_window,
            "detectors": [],
            "evidence": []
        }

    severity_rank = {
        "LOW": 1,
        "MEDIUM": 2,
        "HIGH": 3,
        "CRITICAL": 4
    }

    severity = max(
        (
            r.get("severity", "LOW")
            for r in active
        ),
        key=lambda x: severity_rank.get(x, 0)
    )

    strongest = max(
        active,
        key=lambda r: float(
            r.get("score", 0)
        )
    )

    return {
        "prediction": "THREAT",
        "severity": severity,
        "score": round(
            float(
                strongest["score"]
            ),
            4
        ),
        "primary_threat": strongest[
            "threat_class"
        ],
        "source": source,
        "time_window": time_window,
        "detectors": [
            {
                "detector": r["detector"],
                "threat_class": r[
                    "threat_class"
                ],
                "score": r["score"],
                "severity": r["severity"],
                "evidence": r.get(
                    "evidence",
                    []
                )
            }
            for r in active
        ],
        "detector_count": len(active)
    }


print(" Final alert builder ready")

 Final alert builder ready


In [11]:
# ============================================================
# STEP 310 — GENERATE REAL DDOS API PAYLOAD
# ============================================================

import json
import pandas as pd

# Load exact schema
with open(
    project_root / "models" / "dos_feature_schema.json",
    "r",
    encoding="utf-8"
) as f:
    dos_schema = json.load(f)

DDOS_FEATURES = dos_schema["features"]

print("Required DDoS features:", len(DDOS_FEATURES))

# Load saved feature artifact
ddos_data = pd.read_parquet(
    project_root
    / "data"
    / "processed"
    / "ddos_features.parquet"
)

# Load saved inference results
ddos_pred = pd.read_parquet(
    project_root
    / "data"
    / "processed"
    / "ddos_inference.parquet"
)

# ------------------------------------------------------------
# Find highest-scoring predicted attack
# ------------------------------------------------------------

positive = ddos_pred[
    ddos_pred["prediction"] == 1
].sort_values(
    "model_score",
    ascending=False
)

if positive.empty:
    raise RuntimeError(
        "No predicted DDoS attack found."
    )

selected = positive.iloc[0]

# Match row using time_window
selected_time = selected["time_window"]

row = ddos_data[
    ddos_data["time_window"] == selected_time
]

if row.empty:
    raise RuntimeError(
        "Could not match DDoS inference row "
        "to feature row."
    )

row = row.iloc[0]

# ------------------------------------------------------------
# Verify all 62 features
# ------------------------------------------------------------

missing = [
    feature
    for feature in DDOS_FEATURES
    if feature not in row.index
]

if missing:
    raise RuntimeError(
        "Missing DDoS features: "
        + ", ".join(missing)
    )

# ------------------------------------------------------------
# Build API payload
# ------------------------------------------------------------

payload = {
    "source": None,

    "time_window": str(
        selected_time
    ),

    "ddos_features": {
        feature: float(
            row[feature]
        )
        for feature in DDOS_FEATURES
    }
}

print("\n========================================")
print("REAL DDOS API PAYLOAD")
print("========================================")

print(
    "Selected time:",
    selected_time
)

print(
    "Original model score:",
    float(selected["model_score"])
)

print(
    json.dumps(
        payload,
        indent=2
    )
)

Required DDoS features: 62

REAL DDOS API PAYLOAD
Selected time: 2018-02-21 02:19:14
Original model score: 0.9997705632166489
{
  "source": null,
  "time_window": "2018-02-21 02:19:14",
  "ddos_features": {
    "flow_count": 1589.0,
    "total_packets": 7173.0,
    "total_bytes": 991936.0,
    "mean_flow_duration": 7098.543108873505,
    "median_flow_duration": 6860.0,
    "mean_packets_per_second": 2131.1527087928066,
    "max_packets_per_second": 8588.957055,
    "mean_bytes_per_second": 339595.7261411517,
    "max_bytes_per_second": 1515337.423,
    "mean_packet_size": 89.17881865808684,
    "mean_iat": 5305.178833647766,
    "mean_syn_count": 0.0,
    "mean_ack_count": 0.4971680302076778,
    "mean_rst_count": 0.5028319697923223,
    "flow_count_lag1": 1582.0,
    "flow_count_lag2": 1585.0,
    "flow_count_lag3": 1591.0,
    "flow_count_rolling3": 1586.0,
    "total_packets_lag1": 7159.0,
    "total_packets_lag2": 7170.0,
    "total_packets_lag3": 7182.0,
    "total_packets_rolling